# DCGAN on Fashion-MNIST

A DCGAN (Radford et al., 2016) implementation trained from scratch:
strided-convolution discriminator, transposed-convolution generator,
one-sided label smoothing, Adam (β1=0.5) for both networks.

Run end-to-end, this notebook trains the model, logs loss curves, saves
generated-sample grids over the course of training, and (optionally)
scores the result with FID / Inception Score via Keras InceptionV3.

See [docs/results.md](../docs/results.md) for the actual completed run
(40 epochs, real generated samples and loss curves — not this notebook's
unexecuted default state).


## How to use this notebook
1. **Set the configuration** in the next cell (dataset, epochs, etc.).
2. **Run cells in order**.
3. **After training**, run the evaluation cells to compute FID and Inception Score (optional — downloads ImageNet-pretrained InceptionV3 weights).
4. Generated figures are saved into `outputs/`.


In [ ]:
# ✅ Environment check
import os, math, time, glob, json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
print("GPUs available:", len(tf.config.list_physical_devices('GPU')))

In [ ]:
# ✅ Configuration
DATASET = 'fashion_mnist'  # 'mnist' | 'fashion_mnist' | 'pokemon'
# DATA_DIR = '/content/pokemon_images'  # used only if DATASET == 'pokemon'

# Image shapes: for MNIST/Fashion-MNIST we use 28x28x1; for Pokémon use 64x64x3
IMG_SHAPE = (28, 28, 1) if DATASET in ['mnist', 'fashion_mnist'] else (64, 64, 3)

# Training
BATCH_SIZE = 256 if IMG_SHAPE[0] == 28 else 128
EPOCHS = 40  # matches the completed run in docs/results.md
LATENT_DIM = 100
LEARNING_RATE = 2e-4
BETA_1 = 0.5
LABEL_SMOOTH = 0.9  # one-sided label smoothing for real labels

# Sampling and logging
N_SAMPLE_GRID = 16
SAMPLES_EVERY = 1  # epochs
OUTPUT_DIR = 'outputs'
os.makedirs(os.path.join(OUTPUT_DIR, 'samples'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'checkpoints'), exist_ok=True)

# Reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [ ]:
# ✅ Data loading & preprocessing
def preprocess_to_minus1_1(x):
    x = tf.cast(x, tf.float32)
    return (x - 127.5) / 127.5

if DATASET == 'mnist':
    (train_images, _), _ = tf.keras.datasets.mnist.load_data()
    train_images = train_images[..., np.newaxis]  # (N,28,28,1)
    ds = tf.data.Dataset.from_tensor_slices(train_images)
    ds = ds.map(lambda x: preprocess_to_minus1_1(x), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.shuffle(60000).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

elif DATASET == 'fashion_mnist':
    (train_images, _), _ = tf.keras.datasets.fashion_mnist.load_data()
    train_images = train_images[..., np.newaxis]  # (N,28,28,1)
    ds = tf.data.Dataset.from_tensor_slices(train_images)
    ds = ds.map(lambda x: preprocess_to_minus1_1(x), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.shuffle(60000).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

elif DATASET == 'pokemon':
    # Expecting a folder with images, any nested structure is fine
    # Example: DATA_DIR = '/content/pokemon_images'
    ds = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR, label_mode=None, image_size=IMG_SHAPE[:2],
        color_mode='rgb', batch_size=BATCH_SIZE, shuffle=True)
    ds = ds.map(lambda x: preprocess_to_minus1_1(x), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.unbatch().batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
else:
    raise ValueError("Unknown DATASET selection.")

# Peek a few samples
def show_batch(dataset, n=9):
    it = iter(dataset)
    batch = next(it)
    imgs = batch[:n].numpy()
    if imgs.shape[-1] == 1:
        imgs = np.squeeze(imgs, axis=-1)  # (n, H, W)
        cols = int(np.ceil(np.sqrt(n)))
        rows = int(np.ceil(n / cols))
        plt.figure(figsize=(cols, rows))
        for i in range(n):
            plt.subplot(rows, cols, i+1)
            plt.imshow((imgs[i] + 1.0) * 127.5, cmap='gray')
            plt.axis('off')
        plt.show()
    else:
        cols = int(np.ceil(np.sqrt(n)))
        rows = int(np.ceil(n / cols))
        plt.figure(figsize=(cols, rows))
        for i in range(n):
            plt.subplot(rows, cols, i+1)
            plt.imshow(((imgs[i] + 1.0) * 127.5).astype(np.uint8))
            plt.axis('off')
        plt.show()

show_batch(ds, n=9)

In [ ]:
# ✅ Models: Generator & Discriminator
def make_generator_model(latent_dim, img_shape):
    h, w, c = img_shape
    if (h, w, c) == (28, 28, 1):
        inputs = tf.keras.Input(shape=(latent_dim,), name='z')
        x = layers.Dense(7 * 7 * 256, use_bias=False)(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.LeakyReLU()(x)
        x = layers.Reshape((7, 7, 256))(x)
        x = layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.LeakyReLU()(x)
        x = layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.LeakyReLU()(x)
        outputs = layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh')(x)
        return tf.keras.Model(inputs, outputs, name='generator')
    else:
        inputs = tf.keras.Input(shape=(latent_dim,), name='z')
        x = layers.Dense(4 * 4 * 1024, use_bias=False)(inputs)
        x = layers.BatchNormalization()(x); x = layers.LeakyReLU()(x)
        x = layers.Reshape((4, 4, 1024))(x)
        x = layers.Conv2DTranspose(512, (5, 5), strides=(2, 2), padding='same', use_bias=False)(x)  # 8x8
        x = layers.BatchNormalization()(x); x = layers.LeakyReLU()(x)
        x = layers.Conv2DTranspose(256, (5, 5), strides=(2, 2), padding='same', use_bias=False)(x)  # 16x16
        x = layers.BatchNormalization()(x); x = layers.LeakyReLU()(x)
        x = layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same', use_bias=False)(x)  # 32x32
        x = layers.BatchNormalization()(x); x = layers.LeakyReLU()(x)
        outputs = layers.Conv2DTranspose(img_shape[-1], (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh')(x)  # 64x64
        return tf.keras.Model(inputs, outputs, name='generator')

def make_discriminator_model(img_shape):
    h, w, c = img_shape
    inputs = tf.keras.Input(shape=img_shape, name='img')
    x = inputs
    x = layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same')(x)
    x = layers.LeakyReLU(alpha=0.2)(x); x = layers.Dropout(0.3)(x)
    x = layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same')(x)
    x = layers.LeakyReLU(alpha=0.2)(x); x = layers.Dropout(0.3)(x)
    if (h, w) != (28, 28):  # for 64x64 add a couple extra downsamples
        x = layers.Conv2D(256, (5, 5), strides=(2, 2), padding='same')(x)
        x = layers.LeakyReLU(alpha=0.2)(x); x = layers.Dropout(0.3)(x)
        x = layers.Conv2D(512, (5, 5), strides=(2, 2), padding='same')(x)
        x = layers.LeakyReLU(alpha=0.2)(x); x = layers.Dropout(0.3)(x)
    x = layers.Flatten()(x)
    outputs = layers.Dense(1)(x)  # logits
    return tf.keras.Model(inputs, outputs, name='discriminator')

G = make_generator_model(LATENT_DIM, IMG_SHAPE)
D = make_discriminator_model(IMG_SHAPE)

G.summary(); D.summary()

In [ ]:
# ✅ Losses & Optimizers
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def generator_loss(fake_logits):
    # wants D to classify fakes as real
    return cross_entropy(tf.ones_like(fake_logits), fake_logits)

def discriminator_loss(real_logits, fake_logits, label_smooth=LABEL_SMOOTH):
    real_labels = tf.ones_like(real_logits) * label_smooth
    real_loss = cross_entropy(real_labels, real_logits)
    fake_loss = cross_entropy(tf.zeros_like(fake_logits), fake_logits)
    return real_loss + fake_loss

G_opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, beta_1=BETA_1, beta_2=0.999)
D_opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, beta_1=BETA_1, beta_2=0.999)

# Fixed noise for monitoring progress
fixed_noise = tf.random.normal([N_SAMPLE_GRID, LATENT_DIM])

In [ ]:
# ✅ Utilities (sampling & saving)
import itertools
from IPython.display import clear_output

def make_grid(imgs, nrows=None, ncols=None):
    N = imgs.shape[0]
    if nrows is None or ncols is None:
        ncols = int(np.ceil(np.sqrt(N)))
        nrows = int(np.ceil(N / ncols))
    H, W = imgs.shape[1], imgs.shape[2]
    C = 1 if imgs.ndim == 3 else imgs.shape[3]
    grid = np.zeros((nrows*H, ncols*W, C), dtype=imgs.dtype)
    for idx, img in enumerate(imgs):
        r, c = divmod(idx, ncols)
        grid[r*H:(r+1)*H, c*W:(c+1)*W, ...] = img
    if C == 1:
        grid = grid.squeeze(-1)
    return grid

def sample_and_save(epoch, model, out_dir, n=N_SAMPLE_GRID):
    preds = model(fixed_noise, training=False).numpy()
    # Convert from [-1,1] to [0,255]
    preds_disp = (preds + 1.0) * 127.5
    preds_disp = np.clip(preds_disp, 0, 255).astype(np.uint8)
    grid = make_grid(preds_disp, None, None)
    plt.figure(figsize=(4, 4))
    if grid.ndim == 2:
        plt.imshow(grid, cmap='gray')
    else:
        plt.imshow(grid)
    plt.axis('off')
    fp = os.path.join(out_dir, f'image_epoch_{epoch:04d}.png')
    plt.savefig(fp, bbox_inches='tight', pad_inches=0)
    plt.show()
    return fp

In [ ]:
# ✅ Training loop
train_gen_loss = []
train_disc_loss = []

@tf.function
def train_step(images):
    noise = tf.random.normal([tf.shape(images)[0], LATENT_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated = G(noise, training=True)
        real_logits = D(images, training=True)
        fake_logits = D(generated, training=True)

        g_loss = generator_loss(fake_logits)
        d_loss = discriminator_loss(real_logits, fake_logits)

    grads_g = gen_tape.gradient(g_loss, G.trainable_variables)
    grads_d = disc_tape.gradient(d_loss, D.trainable_variables)
    G_opt.apply_gradients(zip(grads_g, G.trainable_variables))
    D_opt.apply_gradients(zip(grads_d, D.trainable_variables))
    return g_loss, d_loss

def train(ds, epochs=EPOCHS):
    step = 0
    for epoch in range(1, epochs+1):
        tic = time.time()
        for batch in ds:
            g_loss, d_loss = train_step(batch)
            train_gen_loss.append(float(g_loss))
            train_disc_loss.append(float(d_loss))
            step += 1
        if (epoch % SAMPLES_EVERY) == 0:
            clear_output(wait=True)
            fp = sample_and_save(epoch, G, os.path.join(OUTPUT_DIR, 'samples'))
            print(f"[Epoch {epoch}] sample saved to: {fp}")
        print(f"Epoch {epoch:03d} completed in {time.time()-tic:.1f}s | G_loss={train_gen_loss[-1]:.4f} | D_loss={train_disc_loss[-1]:.4f}")
    print("Training complete.")
    # Save final weights
    G.save_weights(os.path.join(OUTPUT_DIR, 'checkpoints', 'G_final.h5'))
    D.save_weights(os.path.join(OUTPUT_DIR, 'checkpoints', 'D_final.h5'))

# ⏱️ Start training
# Uncomment the next line to train when you are ready.
train(ds, EPOCHS)

In [ ]:
# ✅ Plot training curves (run after training)
if len(train_gen_loss) > 0:
    plt.figure()
    plt.plot(train_gen_loss, label='Generator loss')
    plt.plot(train_disc_loss, label='Discriminator loss')
    plt.xlabel('Training steps')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('DCGAN Training Losses')
    plt.savefig(os.path.join(OUTPUT_DIR, 'training_losses.png'), bbox_inches='tight')
    plt.show()
else:
    print("No training history yet. Train first, then re-run this cell.")

In [ ]:
# ✅ Evaluation: Fréchet Inception Distance (FID) & Inception Score (IS)
# Notes:
# - Uses Keras InceptionV3. Internet may be required the first time to download weights.
# - FID is generally more robust than IS for grayscale digits; interpret IS cautiously.

import numpy as np
from scipy import linalg
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input

def _to_inception_input(x):
    # x in [-1,1], shape [N,H,W,C]
    x = (x + 1.0) * 127.5
    if x.shape[-1] == 1:
        x = tf.image.grayscale_to_rgb(x)
    x = tf.image.resize(x, (299, 299))
    x = preprocess_input(x)
    return x

# Global models (created lazily to avoid needless downloads)
_inception_pool = None
_inception_softmax = None

def get_inception_pool_model():
    global _inception_pool
    if _inception_pool is None:
        _inception_pool = InceptionV3(include_top=False, pooling='avg', weights='imagenet')
    return _inception_pool

def get_inception_softmax_model():
    global _inception_softmax
    if _inception_softmax is None:
        _inception_softmax = InceptionV3(include_top=True, weights='imagenet')
    return _inception_softmax

def get_activations(images, batch_size=32):
    model = get_inception_pool_model()
    acts = []
    for i in range(0, images.shape[0], batch_size):
        batch = images[i:i+batch_size]
        batch = _to_inception_input(tf.convert_to_tensor(batch)) # Convert to tensor
        a = model(batch, training=False).numpy()
        acts.append(a)
    return np.concatenate(acts, axis=0)

def calculate_frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):
    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if not np.isfinite(covmean).all():
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))

    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff.dot(diff) + np.trace(sigma1 + sigma2 - 2.0 * covmean)
    return float(fid)

def kl_divergence(p, q, eps=1e-16):
    p = np.clip(p, eps, 1.0); q = np.clip(q, eps, 1.0)
    return np.sum(p * (np.log(p) - np.log(q)))

def inception_score(images, splits=10):
    model = get_inception_softmax_model()
    preds = []
    bs = 32
    for i in range(0, images.shape[0], bs):
        batch = images[i:i+bs]
        batch = _to_inception_input(tf.convert_to_tensor(batch)) # Convert to tensor
        p = model(batch, training=False).numpy()
        preds.append(p)
    preds = np.concatenate(preds, axis=0)
    N = preds.shape[0]
    split_scores = []
    for k in range(splits):
        part = preds[k*N//splits:(k+1)*N//splits]
        py = np.mean(part, axis=0)
        scores = [kl_divergence(part[i], py) for i in range(part.shape[0])]
        split_scores.append(np.exp(np.mean(scores)))
    return float(np.mean(split_scores)), float(np.std(split_scores))

def compute_fid(real_images, fake_images):
    real_acts = get_activations(real_images)
    fake_acts = get_activations(fake_images)
    mu1, sigma1 = real_acts.mean(axis=0), np.cov(real_acts, rowvar=False)
    mu2, sigma2 = fake_acts.mean(axis=0), np.cov(fake_acts, rowvar=False)
    fid = calculate_frechet_distance(mu1, sigma1, mu2, sigma2)
    return fid


# ---- Sample evaluation workflow ----
N = 1000  # adjust for speed vs. accuracy
# # Collect N real images
reals = []
for batch in ds:
     reals.append(batch.numpy())
     if sum(x.shape[0] for x in reals) >= N:
         break
reals = np.concatenate(reals, axis=0)[:N]
# # Generate N fake images
noises = tf.random.normal([N, LATENT_DIM])
fakes = G(noises, training=False).numpy()
#
fid_value = compute_fid(reals, fakes)
is_mean, is_std = inception_score(fakes, splits=10)
print(f"FID: {fid_value:.2f} | Inception Score: {is_mean:.3f} ± {is_std:.3f}")

In [ ]:
# ✅ Export a larger grid of samples for your report (run after training)
def export_samples(n=64, filename='samples_grid.png'):
    z = tf.random.normal([n, LATENT_DIM])
    imgs = G(z, training=False).numpy()
    imgs_disp = (imgs + 1.0) * 127.5
    imgs_disp = np.clip(imgs_disp, 0, 255).astype(np.uint8)
    cols = int(np.ceil(np.sqrt(n)))
    rows = int(np.ceil(n / cols))
    grid = make_grid(imgs_disp, rows, cols)
    plt.figure(figsize=(8, 8))
    if grid.ndim == 2:
        plt.imshow(grid, cmap='gray')
    else:
        plt.imshow(grid)
    plt.axis('off')
    fp = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(fp, bbox_inches='tight', pad_inches=0)
    plt.show()
    print("Saved:", fp)

# Example:
export_samples(n=64, filename='samples_grid.png')

---

## Appendix: Citations & Notes

- Goodfellow, I., Pouget-Abadie, J., Mirza, M., Xu, B., Warde-Farley, D., Ozair, S., Courville, A., & Bengio, Y. (2014). *Generative Adversarial Nets*. NeurIPS.  
- Radford, A., Metz, L., & Chintala, S. (2016). *Unsupervised representation learning with deep convolutional generative adversarial networks*. ICLR.  
- Heusel, M., Ramsauer, H., Unterthiner, T., Nessler, B., & Hochreiter, S. (2017). *GANs Trained by a Two Time-Scale Update Rule Converge to a Local Nash Equilibrium* (Introduces FID). NeurIPS.  
- Salimans, T., Goodfellow, I., Zaremba, W., Cheung, V., Radford, A., & Chen, X. (2016). *Improved Techniques for Training GANs* (Introduces IS). NeurIPS.  
- TensorFlow (2024). *Deep Convolutional GAN Tutorial*. https://www.tensorflow.org/tutorials/generative/dcgan  

> Be sure to include APA 7th citations in your **report**. If you use this notebook or an AI assistant to draft text, cite it clearly per course policy.